In [10]:
from pathlib import Path
from jobspy import scrape_jobs
import pandas as pd
from IPython.display import HTML, display

# https://github.com/speedyapply/JobSpy
jobs = scrape_jobs(
    site_name=["indeed", "linkedin"],
    linkedin_fetch_description=True,
    search_term='"AI Engineer"',
    location="USA",
    country_indeed="USA",
    job_type="fulltime",
    hours_old=72,
    results_wanted=20, 
)

df = pd.DataFrame(jobs)

output_dir = Path("Job Posting Data")
output_dir.mkdir(parents=True, exist_ok=True)
jsonl_path = output_dir / "1-scraped_jobs.jsonl"

# We ensure the title contains BOTH "AI" and "Engineer" but not "Intern"
mask = (
    df['title'].str.contains('AI', case=False, na=False)
    & df['title'].str.contains('Engineer', case=False, na=False)
    & ~df['title'].str.contains('Intern', case=False, na=False)
)
matching_jobs = df[mask].copy()

# We drop duplicates based on the combination of 'title' and 'company'
filtered_jobs = matching_jobs.drop_duplicates(subset=['title', 'company']).copy()
filtered_jobs.to_json(jsonl_path, orient='records', lines=True, force_ascii=False)

print(f"Saved filtered jobs to: {jsonl_path.resolve()}")
print(f"Total jobs found: {len(df)}")
print(f"Jobs matching 'AI' + 'Engineer' in title: {len(matching_jobs)}")
print(f"Jobs after deduplicating identical title/company pairs: {len(filtered_jobs)}")

results_to_show = filtered_jobs[['title', 'company', 'job_url']].copy()
description_series = filtered_jobs['description'] if 'description' in filtered_jobs.columns else pd.Series('', index=filtered_jobs.index)
results_to_show['description_chars'] = description_series.fillna('').astype(str).str.len()
results_to_show = results_to_show[['title', 'company', 'description_chars', 'job_url']]
results_to_show['job_url'] = results_to_show['job_url'].apply(
    lambda url: f'<a href=\"{url}\" target=\"_blank\">{url}</a>' if pd.notna(url) else ''
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

Saved filtered jobs to: /Users/lukaslechner/PythonProjects/ai-engineering-foundations-labs/1-introduction/Job Posting Data/1-scraped_jobs.jsonl
Total jobs found: 40
Jobs matching 'AI' + 'Engineer' in title: 27
Jobs after deduplicating identical title/company pairs: 26


title,company,description_chars,job_url
AI Engineer (DevOps),Wipro,3275,https://www.indeed.com/viewjob?jk=f827c33d4ee013f2
Principal AI Engineer,VideoAmp,6285,https://www.indeed.com/viewjob?jk=a59140bb24e25483
AI Engineer / Agentic and Generative AI,Realign,2957,https://www.indeed.com/viewjob?jk=dd8d94bee2e51705
Software AI Engineer – Agentic AI-2,Realign,2968,https://www.indeed.com/viewjob?jk=8ac7b54fb8ca57c0
"AI Engineer, Enterprise AI (Principal/Sr. Principal/Distinguished)",Palo Alto Networks,6890,https://www.indeed.com/viewjob?jk=07564be4ba2a6d92
Junior Applied AI Engineer- HRIS,Fortinet,4095,https://www.indeed.com/viewjob?jk=19fc4813887bb16d
Microservices Tech Lead – AI Engineer/Analyst - Vice President – TAMPA,Information Technology Senior Management Forum,4401,https://www.indeed.com/viewjob?jk=911b6e2b9ebbe41b
Principal AI Engineer,Burrell Behavioral Health,6831,https://www.indeed.com/viewjob?jk=09230ed99d1cba66
"Principal, Software Engineer - GenAi",Walmart,8271,https://www.indeed.com/viewjob?jk=345ca7e57ef93de2
AI Engineer - AI Department,Petlibro,5143,https://www.linkedin.com/jobs/view/4396463746
